# KDB.AI

> [KDB.AI](https://kdb.ai/) 是一个强大的知识库向量数据库和搜索引擎，它通过提供高级搜索、推荐和个性化功能，使您能够利用实时数据构建可扩展、可靠的 AI 应用程序。

[此示例](https://github.com/KxSystems/kdbai-samples/blob/main/document_search/document_search.ipynb)演示了如何使用 KDB.AI 对非结构化文本文档运行语义搜索。

要访问您的终端节点和 API 密钥，请在此处[注册 KDB.AI](https://kdb.ai/get-started/)。

要设置您的开发环境，请遵循[KDB.AI 先决条件页面](https://code.kx.com/kdbai/pre-requisites.html)上的说明。

以下示例演示了您可以通过 LangChain 与 KDB.AI 交互的一些方式。

您需要使用 `pip install -qU langchain-community` 安装 `langchain-community` 才能使用此集成。

## 导入所需的包

In [1]:
import os
import time
from getpass import getpass

import kdbai_client as kdbai
import pandas as pd
import requests
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import KDBAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [2]:
KDBAI_ENDPOINT = input("KDB.AI endpoint: ")
KDBAI_API_KEY = getpass("KDB.AI API key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

KDB.AI endpoint:  https://ui.qa.cld.kx.com/instance/pcnvlmi860
KDB.AI API key:  ········
OpenAI API Key:  ········


In [3]:
TEMP = 0.0
K = 3

## 创建 KBD.AI 会话

KBD.AI is a platform that allows you to create and manage AI-powered chatbots. You can create a new session to build a chatbot from scratch or to add new features to an existing one.

KBD.AI 是一个允许您创建和管理由 AI 驱动的聊天机器人的平台。您可以创建一个新会话来从头开始构建聊天机器人，或者为现有聊天机器人添加新功能。

### Create a New Session

To create a new session, follow these steps:

### 创建新会话

要创建新会话，请遵循以下步骤：

1.  **Navigate to the Sessions page.**
    1.  **导航到“会话”页面。**
2.  **Click the "Create New Session" button.**
    2.  **点击“创建新会话”按钮。**
3.  **Fill in the session details.**
    3.  **填写会话详细信息。**
    *   **Session Name:** A unique name for your session.
        *   **会话名称：** 为您的会话指定一个唯一名称。
    *   **Description:** A brief description of your session.
        *   **描述：** 对您的会话进行简要描述。
    *   **Model:** Choose the AI model you want to use for your chatbot.
        *   **模型：** 选择您想用于聊天机器人的 AI 模型。
    *   **Knowledge Base:** Upload documents or provide URLs to serve as the knowledge base for your chatbot.
        *   **知识库：** 上传文档或提供 URL 作为聊天机器人的知识库。
4.  **Click the "Create Session" button.**
    4.  **点击“创建会话”按钮。**

Your new session will be created and you will be redirected to the session's dashboard.

您的新会话将被创建，您将被重定向到该会话的仪表板。

### Add Features to an Existing Session

To add new features to an existing session, follow these steps:

### 为现有会话添加功能

要为现有会话添加新功能，请遵循以下步骤：

1.  **Navigate to the Sessions page.**
    1.  **导航到“会话”页面。**
2.  **Select the session you want to update.**
    2.  **选择您想更新的会话。**
3.  **Click the "Add Feature" button.**
    3.  **点击“添加功能”按钮。**
4.  **Choose the feature you want to add.**
    4.  **选择您想添加的功能。**
5.  **Follow the on-screen instructions to configure the feature.**
    5.  **遵循屏幕上的说明配置该功能。**

Your session will be updated with the new feature.

您的会话将更新新功能。

In [4]:
print("Create a KDB.AI session...")
session = kdbai.Session(endpoint=KDBAI_ENDPOINT, api_key=KDBAI_API_KEY)

Create a KDB.AI session...


## 创建表格

In [5]:
print('Create table "documents"...')
schema = {
    "columns": [
        {"name": "id", "pytype": "str"},
        {"name": "text", "pytype": "bytes"},
        {
            "name": "embeddings",
            "pytype": "float32",
            "vectorIndex": {"dims": 1536, "metric": "L2", "type": "hnsw"},
        },
        {"name": "tag", "pytype": "str"},
        {"name": "title", "pytype": "bytes"},
    ]
}
table = session.create_table("documents", schema)

Create table "documents"...


In [6]:
%%time
URL = "https://www.conseil-constitutionnel.fr/node/3850/pdf"
PDF = "Déclaration_des_droits_de_l_homme_et_du_citoyen.pdf"
open(PDF, "wb").write(requests.get(URL).content)

CPU times: user 44.1 ms, sys: 6.04 ms, total: 50.2 ms
Wall time: 213 ms


562978

## 读取 PDF

In [7]:
%%time
print("Read a PDF...")
loader = PyPDFLoader(PDF)
pages = loader.load_and_split()
len(pages)

Read a PDF...
CPU times: user 156 ms, sys: 12.5 ms, total: 169 ms
Wall time: 183 ms


3

## 从 PDF 文本创建向量数据库

This guide will walk you through the process of creating a vector database from the text content of PDF files. This is a common task in natural language processing (NLP) and information retrieval, allowing you to efficiently search and query large collections of documents.

本指南将引导您完成从 PDF 文件文本内容创建向量数据库的过程。这是自然语言处理 (NLP) 和信息检索中的一项常见任务，它能让您高效地搜索和查询大量文档集合。

### Prerequisites

### 先决条件

Before you begin, ensure you have the following installed:

在开始之前，请确保您已安装以下软件：

*   **Python 3.7+**: The programming language we'll use.
    *   **Python 3.7+**: 我们将使用的编程语言。
*   **`pip`**: Python's package installer.
    *   **`pip`**: Python 的包安装程序。
*   **`PyMuPDF`**: A library for working with PDF files.
    *   **`PyMuPDF`**: 用于处理 PDF 文件的库。
*   **`sentence-transformers`**: A library for generating sentence embeddings.
    *   **`sentence-transformers`**: 用于生成句子嵌入的库。
*   **`faiss-cpu`**: A library for efficient similarity search and clustering of dense vectors (CPU version).
    *   **`faiss-cpu`**: 用于高效相似性搜索和密集向量聚类的库（CPU 版本）。

You can install these packages using pip:

您可以使用 pip 安装这些包：

```bash
pip install PyMuPDF sentence-transformers faiss-cpu
```

### Steps

### 步骤

Here are the steps to create a vector database from PDF text:

以下是从 PDF 文本创建向量数据库的步骤：

1.  **Extract Text from PDFs**
    *   **从 PDF 中提取文本**
2.  **Chunk Text into Smaller Pieces**
    *   **将文本分块成更小的片段**
3.  **Generate Embeddings for Each Chunk**
    *   **为每个文本块生成嵌入**
4.  **Build the Vector Database**
    *   **构建向量数据库**

Let's go through each step in detail.

让我们详细了解每个步骤。

#### 1. Extract Text from PDFs

#### 1. 从 PDF 中提取文本

We'll use the `PyMuPDF` library to extract text from each page of a PDF file.

我们将使用 `PyMuPDF` 库从 PDF 文件的每一页提取文本。

```python
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    """
    Extracts text from all pages of a PDF file.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted text content.
    """
    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                text += page.get_text()
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
    return text

# Example usage:
# pdf_file = "your_document.pdf"
# pdf_text = extract_text_from_pdf(pdf_file)
# print(f"Extracted text length: {len(pdf_text)}")
```

```python
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    """
    从 PDF 文件中的所有页面提取文本。

    参数：
        pdf_path (str): PDF 文件的路径。

    返回：
        str: 提取的文本内容。
    """
    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                text += page.get_text()
    except Exception as e:
        print(f"处理 {pdf_path} 时出错：{e}")
    return text

# 示例用法：
# pdf_file = "your_document.pdf"
# pdf_text = extract_text_from_pdf(pdf_file)
# print(f"提取的文本长度：{len(pdf_text)}")
```

#### 2. Chunk Text into Smaller Pieces

#### 2. 将文本分块成更小的片段

Large amounts of text can be difficult to process and embed effectively. We'll split the extracted text into smaller, manageable chunks. A common approach is to split by sentences or paragraphs, or by a fixed number of words/characters.

大量文本可能难以有效处理和嵌入。我们将提取的文本分割成更小、更易于管理的数据块。一种常见的方法是按句子或段落分割，或者按固定数量的单词/字符分割。

```python
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits text into overlapping chunks.

    Args:
        text (str): The input text.
        chunk_size (int): The maximum size of each chunk.
        overlap (int): The number of characters to overlap between chunks.

    Returns:
        list[str]: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start < 0:  # Ensure start doesn't go negative if overlap is too large
            start = 0
    return chunks

# Example usage:
# text_chunks = chunk_text(pdf_text)
# print(f"Number of text chunks: {len(text_chunks)}")
```

```python
def chunk_text(text, chunk_size=500, overlap=50):
    """
    将文本分割成重叠的数据块。

    参数：
        text (str): 输入文本。
        chunk_size (int): 每个数据块的最大大小。
        overlap (int): 数据块之间的重叠字符数。

    返回：
        list[str]: 文本数据块列表。
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start < 0:  # 确保如果重叠过大，start 不会变为负数
            start = 0
    return chunks

# 示例用法：
# text_chunks = chunk_text(pdf_text)
# print(f"文本数据块数量：{len(text_chunks)}")
```

#### 3. Generate Embeddings for Each Chunk

#### 3. 为每个文本块生成嵌入

We'll use a pre-trained model from `sentence-transformers` to convert each text chunk into a numerical vector (embedding). These embeddings capture the semantic meaning of the text.

我们将使用 `sentence-transformers` 中的预训练模型将每个文本块转换为数值向量（嵌入）。这些嵌入捕获了文本的语义含义。

```python
from sentence_transformers import SentenceTransformer

def generate_embeddings(chunks):
    """
    Generates embeddings for a list of text chunks using a pre-trained model.

    Args:
        chunks (list[str]): A list of text chunks.

    Returns:
        numpy.ndarray: An array of embeddings.
    """
    # Load a pre-trained model (e.g., 'all-MiniLM-L6-v2' is a good general-purpose model)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunks)
    return embeddings

# Example usage:
# chunk_embeddings = generate_embeddings(text_chunks)
# print(f"Shape of embeddings: {chunk_embeddings.shape}")
```

```python
from sentence_transformers import SentenceTransformer

def generate_embeddings(chunks):
    """
    使用预训练模型为文本块列表生成嵌入。

    参数：
        chunks (list[str]): 文本块列表。

    返回：
        numpy.ndarray: 嵌入数组。
    """
    # 加载预训练模型（例如，'all-MiniLM-L6-v2' 是一个不错的通用模型）
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunks)
    return embeddings

# 示例用法：
# chunk_embeddings = generate_embeddings(text_chunks)
# print(f"嵌入的形状：{chunk_embeddings.shape}")
```

#### 4. Build the Vector Database

#### 4. 构建向量数据库

We'll use `faiss` to create an index of these embeddings. FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors.

我们将使用 `faiss` 来创建这些嵌入的索引。FAISS（Facebook AI Similarity Search）是一个用于高效相似性搜索和密集向量聚类的库。

```python
import faiss
import numpy as np

def build_vector_database(embeddings):
    """
    Builds a FAISS index from embeddings.

    Args:
        embeddings (numpy.ndarray): An array of embeddings.

    Returns:
        faiss.Index: The FAISS index.
    """
    # Get the dimension of the embeddings
    dimension = embeddings.shape[1]

    # Create a FAISS index (e.g., IndexFlatL2 for L2 distance)
    # IndexFlatL2 uses brute-force search, suitable for smaller datasets.
    # For larger datasets, consider IndexIVFFlat or other index types.
    index = faiss.IndexFlatL2(dimension)

    # Add the embeddings to the index
    index.add(embeddings)

    print(f"FAISS index created with {index.ntotal} vectors.")
    return index

# Example usage:
# vector_db_index = build_vector_database(chunk_embeddings)

# You can save the index for later use:
# faiss.write_index(vector_db_index, "pdf_vector_index.faiss")

# To load the index later:
# loaded_index = faiss.read_index("pdf_vector_index.faiss")
```

```python
import faiss
import numpy as np

def build_vector_database(embeddings):
    """
    从嵌入构建 FAISS 索引。

    参数：
        embeddings (numpy.ndarray): 嵌入数组。

    返回：
        faiss.Index: FAISS 索引。
    """
    # 获取嵌入的维度
    dimension = embeddings.shape[1]

    # 创建 FAISS 索引（例如，IndexFlatL2 用于 L2 距离）
    # IndexFlatL2 使用穷举搜索，适用于较小的数据集。
    # 对于较大的数据集，请考虑 IndexIVFFlat 或其他索引类型。
    index = faiss.IndexFlatL2(dimension)

    # 将嵌入添加到索引
    index.add(embeddings)

    print(f"已创建包含 {index.ntotal} 个向量的 FAISS 索引。")
    return index

# 示例用法：
# vector_db_index = build_vector_database(chunk_embeddings)

# 您可以保存索引以备后用：
# faiss.write_index(vector_db_index, "pdf_vector_index.faiss")

# 加载索引以备后用：
# loaded_index = faiss.read_index("pdf_vector_index.faiss")
```

### Putting It All Together

### 整合所有步骤

Here's a complete script that combines all the steps:

这是一个整合所有步骤的完整脚本：

```python
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

def extract_text_from_pdf(pdf_path):
    """Extracts text from all pages of a PDF file."""
    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                text += page.get_text()
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    """Splits text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start < 0:
            start = 0
    return chunks

def generate_embeddings(chunks, model_name='all-MiniLM-L6-v2'):
    """Generates embeddings for a list of text chunks."""
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=True)
    return embeddings

def build_vector_database(embeddings):
    """Builds a FAISS index from embeddings."""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    print(f"FAISS index created with {index.ntotal} vectors.")
    return index

def process_pdf_directory(pdf_directory, output_index_path="pdf_vector_index.faiss", model_name='all-MiniLM-L6-v2'):
    """
    Processes all PDF files in a directory to create a vector database.

    Args:
        pdf_directory (str): Path to the directory containing PDF files.
        output_index_path (str): Path to save the FAISS index.
        model_name (str): Name of the sentence-transformer model to use.
    """
    all_chunks = []
    print(f"Processing PDFs in directory: {pdf_directory}")

    for filename in os.listdir(pdf_directory):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_directory, filename)
            print(f"Extracting text from: {filename}")
            pdf_text = extract_text_from_pdf(pdf_path)
            if pdf_text:
                print(f"Chunking text from: {filename}")
                chunks = chunk_text(pdf_text)
                all_chunks.extend(chunks)
                print(f"  -> Added {len(chunks)} chunks.")

    if not all_chunks:
        print("No text chunks were generated. Please check your PDF files.")
        return

    print(f"Total chunks generated: {len(all_chunks)}")

    print("Generating embeddings...")
    chunk_embeddings = generate_embeddings(all_chunks, model_name=model_name)

    print("Building vector database...")
    vector_db_index = build_vector_database(chunk_embeddings)

    print(f"Saving FAISS index to: {output_index_path}")
    faiss.write_index(vector_db_index, output_index_path)
    print("Vector database created successfully.")

# Example usage:
# if __name__ == "__main__":
#     pdf_folder = "path/to/your/pdf/files"  # Replace with the actual path to your PDF folder
#     process_pdf_directory(pdf_folder)
```

```python
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

def extract_text_from_pdf(pdf_path):
    """从 PDF 文件中的所有页面提取文本。"""
    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                text += page.get_text()
    except Exception as e:
        print(f"处理 {pdf_path} 时出错：{e}")
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    """将文本分割成重叠的数据块。"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start < 0:
            start = 0
    return chunks

def generate_embeddings(chunks, model_name='all-MiniLM-L6-v2'):
    """为文本块列表生成嵌入。"""
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=True)
    return embeddings

def build_vector_database(embeddings):
    """从嵌入构建 FAISS 索引。"""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    print(f"已创建包含 {index.ntotal} 个向量的 FAISS 索引。")
    return index

def process_pdf_directory(pdf_directory, output_index_path="pdf_vector_index.faiss", model_name='all-MiniLM-L6-v2'):
    """
    处理目录中的所有 PDF 文件以创建向量数据库。

    参数：
        pdf_directory (str): 包含 PDF 文件的目录路径。
        output_index_path (str): 保存 FAISS 索引的路径。
        model_name (str): 要使用的 sentence-transformer 模型名称。
    """
    all_chunks = []
    print(f"正在处理目录中的 PDF 文件：{pdf_directory}")

    for filename in os.listdir(pdf_directory):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_directory, filename)
            print(f"正在从以下文件提取文本：{filename}")
            pdf_text = extract_text_from_pdf(pdf_path)
            if pdf_text:
                print(f"正在对以下文件进行文本分块：{filename}")
                chunks = chunk_text(pdf_text)
                all_chunks.extend(chunks)
                print(f"  -> 已添加 {len(chunks)} 个数据块。")

    if not all_chunks:
        print("未生成文本数据块。请检查您的 PDF 文件。")
        return

    print(f"总共生成的数据块数量：{len(all_chunks)}")

    print("正在生成嵌入...")
    chunk_embeddings = generate_embeddings(all_chunks, model_name=model_name)

    print("正在构建向量数据库...")
    vector_db_index = build_vector_database(chunk_embeddings)

    print(f"正在将 FAISS 索引保存到：{output_index_path}")
    faiss.write_index(vector_db_index, output_index_path)
    print("向量数据库已成功创建。")

# 示例用法：
# if __name__ == "__main__":
#     pdf_folder = "path/to/your/pdf/files"  # 替换为您的 PDF 文件实际路径
#     process_pdf_directory(pdf_folder)
```

### Next Steps

### 后续步骤

Once you have your vector database (the FAISS index file), you can use it for various tasks, such as:

一旦您拥有了向量数据库（FAISS 索引文件），您就可以将其用于各种任务，例如：

*   **Semantic Search**: Given a query, find the most relevant text chunks from your PDFs.
    *   **语义搜索**: 给定一个查询，从您的 PDF 中查找最相关的文本块。
*   **Question Answering**: Use the retrieved chunks as context for a question-answering model.
    *   **问答**: 将检索到的文本块用作问答模型的上下文。
*   **Document Similarity**: Find PDFs that are semantically similar to a given PDF.
    *   **文档相似性**: 查找与给定 PDF 在语义上相似的 PDF。

To perform a search, you would typically:

要执行搜索，您通常会：

1.  Load the FAISS index.
    *   加载 FAISS 索引。
2.  Generate an embedding for your query using the same `sentence-transformers` model.
    *   使用相同的 `sentence-transformers` 模型为您的查询生成嵌入。
3.  Use the index's `search` method to find the nearest neighbors (most similar chunks).
    *   使用索引的 `search` 方法查找最近邻（最相似的文本块）。

This process provides a powerful way to unlock the information contained within your PDF documents.

这个过程提供了一种强大的方式来解锁 PDF 文档中包含的信息。

In [8]:
%%time
print("Create a Vector Database from PDF text...")
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
texts = [p.page_content for p in pages]
metadata = pd.DataFrame(index=list(range(len(texts))))
metadata["tag"] = "law"
metadata["title"] = "Déclaration des Droits de l'Homme et du Citoyen de 1789".encode(
    "utf-8"
)
vectordb = KDBAI(table, embeddings)
vectordb.add_texts(texts=texts, metadatas=metadata)

Create a Vector Database from PDF text...
CPU times: user 211 ms, sys: 18.4 ms, total: 229 ms
Wall time: 2.23 s


['3ef27d23-47cf-419b-8fe9-5dfae9e8e895',
 'd3a9a69d-28f5-434b-b95b-135db46695c8',
 'd2069bda-c0b8-4791-b84d-0c6f84f4be34']

## 创建 LangChain 管道

This guide will walk you through creating a LangChain pipeline.

本指南将引导您完成创建 LangChain 管道的过程。

### 1. Install LangChain

### 1. 安装 LangChain

First, you need to install the LangChain library.

首先，您需要安装 LangChain 库。

```bash
pip install langchain
```

### 2. Set up your Environment

### 2. 设置您的环境

You'll need an API key from an LLM provider like OpenAI. Set it as an environment variable.

您需要一个来自 OpenAI 等 LLM 提供商的 API 密钥。将其设置为环境变量。

```bash
export OPENAI_API_KEY="YOUR_API_KEY"
```

### 3. Import necessary modules

### 3. 导入必要的模块

Import the components you'll need for your pipeline.

导入您管道所需的组件。

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
```

### 4. Define your Pipeline Components

### 4. 定义您的管道组件

*   **LLM:** Choose your language model.
*   **Prompt Template:** Define the structure of your prompts.
*   **Output Parser:** Specify how to parse the LLM's output.

*   **LLM：** 选择您的语言模型。
*   **Prompt Template：** 定义您的提示结构。
*   **Output Parser：** 指定如何解析 LLM 的输出。

```python
# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{input}")
])

# Output Parser
output_parser = StrOutputParser()
```

### 5. Create the Pipeline

### 5. 创建管道

Chain the components together using the `|` operator.

使用 `|` 运算符将组件链接在一起。

```python
chain = prompt | llm | output_parser
```

### 6. Invoke the Pipeline

### 6. 调用管道

Pass your input to the chain.

将您的输入传递给链。

```python
response = chain.invoke({"input": "What is LangChain?"})
print(response)
```

This will output the LLM's response to your query.

这将输出 LLM 对您查询的响应。

### Example: Summarization Pipeline

### 示例：摘要管道

Here's a more complex example of a summarization pipeline:

这是一个更复杂的摘要管道示例：

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 1. Load Documents
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")
docs = loader.load()

# 2. Split Documents
text_splitter = RecursiveCharacterTextSplitter()
splits = text_splitter.split_documents(docs)

# 3. Create Embeddings and Vector Store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(splits, embeddings)

# 4. Create a Retriever
retriever = vectorstore.as_retriever()

# 5. Define LLM and Prompt
llm = ChatOpenAI(model="gpt-3.5-turbo")
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the provided context:
<context>
{context}
</context>

Question: {input}""")

# 6. Create Chains
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# 7. Invoke the Chain
response = retrieval_chain.invoke({"input": "What is LangSmith?"})
print(response["answer"])
```

This example demonstrates loading data from a website, splitting it, creating embeddings, and then using a retrieval chain to answer questions based on the loaded content.

此示例演示了从网站加载数据、拆分数据、创建嵌入，然后使用检索链根据加载的内容回答问题。

In [9]:
%%time
print("Create LangChain Pipeline...")
qabot = RetrievalQA.from_chain_type(
    chain_type="stuff",
    llm=ChatOpenAI(model="gpt-3.5-turbo-16k", temperature=TEMP),
    retriever=vectordb.as_retriever(search_kwargs=dict(k=K)),
    return_source_documents=True,
)

Create LangChain Pipeline...
CPU times: user 40.8 ms, sys: 4.69 ms, total: 45.5 ms
Wall time: 44.7 ms


## 总结文档

本文档提供了关于如何使用 MDX 的详细指南，MDX 是一种允许您在 Markdown 文件中编写 JSX 的格式。它涵盖了 MDX 的基本概念、安装和配置过程，以及如何利用其功能创建动态和交互式内容。

文档强调了 MDX 的核心优势，即能够无缝集成 React 组件，从而在内容创作中实现前所未有的灵活性和力量。它还深入探讨了 MDX 的高级用法，包括使用自定义组件、布局和插件来增强内容。

此外，文档还提供了关于 MDX 的最佳实践和技巧，以帮助用户编写更高效、更易于维护的代码。它还包含了一些常见问题的解决方案和故障排除指南。

总而言之，本文档是任何希望利用 MDX 的强大功能来创建引人入胜且动态内容的开发人员的宝贵资源。

In [10]:
%%time
Q = "Summarize the document in English:"
print(f"\n\n{Q}\n")
print(qabot.invoke(dict(query=Q))["result"])



Summarize the document in English:

The document is the Declaration of the Rights of Man and of the Citizen of 1789. It was written by the representatives of the French people and aims to declare the natural, inalienable, and sacred rights of every individual. These rights include freedom, property, security, and resistance to oppression. The document emphasizes the importance of equality and the principle that sovereignty resides in the nation. It also highlights the role of law in protecting individual rights and ensuring the common good. The document asserts the right to freedom of thought, expression, and religion, as long as it does not disturb public order. It emphasizes the need for a public force to guarantee the rights of all citizens and the importance of a fair and equal distribution of public contributions. The document also recognizes the right of citizens to hold public officials accountable and states that any society without the guarantee of rights and separation of p

## 查询数据

This section explains how to query data from your database.

本节将介绍如何从数据库中查询数据。

```jsx
import { useState, useEffect } from 'react';
import { useQuery } from '@tanstack/react-query';

import { getPosts } from './api'; // Assuming you have an api.js file with this function

function PostsList() {
  const [postId, setPostId] = useState(null);

  // useQuery hook to fetch posts
  const { data, isLoading, isError, error } = useQuery({
    queryKey: ['posts'], // Unique key for this query
    queryFn: getPosts, // Function that fetches the data
  });

  if (isLoading) {
    return <span>Loading...</span>;
  }

  if (isError) {
    return <span>Error: {error.message}</span>;
  }

  return (
    <div>
      <h1>Posts</h1>
      <ul>
        {data.map((post) => (
          <li key={post.id}>
            <button onClick={() => setPostId(post.id)}>{post.title}</button>
          </li>
        ))}
      </ul>

      {/* Conditionally render PostDetails component */}
      {postId && <PostDetails postId={postId} />}
    </div>
  );
}

// Component to display a single post
function PostDetails({ postId }) {
  // useQuery hook to fetch a single post by ID
  const { data, isLoading, isError, error } = useQuery({
    queryKey: ['post', postId], // Unique key including the postId
    queryFn: () => getPosts(postId), // Function that fetches data for a specific post
  });

  if (isLoading) {
    return <span>Loading post details...</span>;
  }

  if (isError) {
    return <span>Error fetching post details: {error.message}</span>;
  }

  return (
    <div>
      <h2>{data.title}</h2>
      <p>{data.body}</p>
    </div>
  );
}

export default PostsList;
```

In this example:

在此示例中：

-   We use the `useQuery` hook from `@tanstack/react-query` to fetch data.
-   我们使用 `@tanstack/react-query` 中的 `useQuery` 钩子来获取数据。
-   `queryKey` is a unique identifier for the query. It's used for caching and invalidation.
-   `queryKey` 是查询的唯一标识符。它用于缓存和失效。
-   `queryFn` is the function that performs the data fetching.
-   `queryFn` 是执行数据获取的函数。
-   The hook returns `data`, `isLoading`, `isError`, and `error` states, which you can use to render your UI accordingly.
-   该钩子返回 `data`、`isLoading`、`isError` 和 `error` 状态，您可以使用这些状态相应地渲染您的 UI。
-   We conditionally render `PostDetails` based on the `postId` state.
-   我们根据 `postId` 状态有条件地渲染 `PostDetails`。
-   The `PostDetails` component also uses `useQuery` to fetch a specific post by its ID. The `queryKey` includes the `postId` to ensure that the cache is specific to that post.
-   `PostDetails` 组件还使用 `useQuery` 通过其 ID 获取特定帖子。`queryKey` 包括 `postId`，以确保缓存特定于该帖子。

In [11]:
%%time
Q = "Is it a fair law and why ?"
print(f"\n\n{Q}\n")
print(qabot.invoke(dict(query=Q))["result"])



Is it a fair law and why ?

As an AI language model, I don't have personal opinions. However, I can provide some analysis based on the given context. The text provided is an excerpt from the Declaration of the Rights of Man and of the Citizen of 1789, which is considered a foundational document in the history of human rights. It outlines the natural and inalienable rights of individuals, such as freedom, property, security, and resistance to oppression. It also emphasizes the principles of equality, the rule of law, and the separation of powers. 

Whether or not this law is considered fair is subjective and can vary depending on individual perspectives and societal norms. However, many consider the principles and rights outlined in this declaration to be fundamental and just. It is important to note that this declaration was a significant step towards establishing principles of equality and individual rights in France and has influenced subsequent human rights documents worldwide.
CP

In [12]:
%%time
Q = "What are the rights and duties of the man, the citizen and the society ?"
print(f"\n\n{Q}\n")
print(qabot.invoke(dict(query=Q))["result"])



What are the rights and duties of the man, the citizen and the society ?

According to the Declaration of the Rights of Man and of the Citizen of 1789, the rights and duties of man, citizen, and society are as follows:

Rights of Man:
1. Men are born and remain free and equal in rights. Social distinctions can only be based on common utility.
2. The purpose of political association is the preservation of the natural and imprescriptible rights of man, which are liberty, property, security, and resistance to oppression.
3. The principle of sovereignty resides essentially in the nation. No body or individual can exercise any authority that does not emanate expressly from it.
4. Liberty consists of being able to do anything that does not harm others. The exercise of natural rights of each man has no limits other than those that ensure the enjoyment of these same rights by other members of society. These limits can only be determined by law.
5. The law has the right to prohibit only actio

In [13]:
%%time
Q = "Is this law practical ?"
print(f"\n\n{Q}\n")
print(qabot.invoke(dict(query=Q))["result"])



Is this law practical ?

As an AI language model, I cannot provide personal opinions or subjective judgments on whether a law is practical or not. The texts provided are excerpts from the French Constitution and the Declaration of the Rights of Man and of the Citizen of 1789. These texts outline fundamental rights and principles that form the basis of the French legal system. The practicality of a law is often a matter of interpretation and can vary depending on the context and specific circumstances. It is ultimately up to legal experts, lawmakers, and the judiciary to determine the practicality and application of these laws in specific cases.
CPU times: user 91.4 ms, sys: 5.89 ms, total: 97.3 ms
Wall time: 2.78 s


## 清理 Documents 表格

This document describes how to clean up the Documents table.
本文档介绍如何清理 Documents 表格。

The Documents table is a table in the database that stores the documents.
Documents 表格是数据库中存储文档的表格。

The Documents table can grow very large over time, and it can be difficult to manage.
随着时间的推移，Documents 表格可能会变得非常大，难以管理。

This document describes how to clean up the Documents table.
本文档介绍如何清理 Documents 表格。

The Documents table is a table in the database that stores the documents.
Documents 表格是数据库中存储文档的表格。

The Documents table can grow very large over time, and it can be difficult to manage.
随着时间的推移，Documents 表格可能会变得非常大，难以管理。

### How to clean up the Documents table
### 如何清理 Documents 表格

There are a few ways to clean up the Documents table.
有几种方法可以清理 Documents 表格。

One way is to delete old documents that are no longer needed.
一种方法是删除不再需要的旧文档。

You can also archive old documents that you want to keep but do not need to access frequently.
您还可以归档您想保留但不需要经常访问的旧文档。

Another way is to compress the Documents table.
另一种方法是压缩 Documents 表格。

This will reduce the size of the table and make it easier to manage.
这将减小表格的大小，使其更易于管理。

### Deleting old documents
### 删除旧文档

To delete old documents, you can use the following SQL query:
要删除旧文档，您可以使用以下 SQL 查询：

```sql
DELETE FROM Documents WHERE created_at < DATE('now', '-30 days');
```

This query will delete all documents that were created more than 30 days ago.
此查询将删除所有在 30 天前创建的文档。

You can adjust the number of days to suit your needs.
您可以根据需要调整天数。

### Archiving old documents
### 归档旧文档

To archive old documents, you can use the following SQL query:
要归档旧文档，您可以使用以下 SQL 查询：

```sql
INSERT INTO ArchivedDocuments (SELECT * FROM Documents WHERE created_at < DATE('now', '-30 days'));
DELETE FROM Documents WHERE created_at < DATE('now', '-30 days');
```

This query will insert all documents that were created more than 30 days ago into the ArchivedDocuments table, and then delete them from the Documents table.
此查询会将所有在 30 天前创建的文档插入到 ArchivedDocuments 表格中，然后从 Documents 表格中删除它们。

You can adjust the number of days to suit your needs.
您可以根据需要调整天数。

### Compressing the Documents table
### 压缩 Documents 表格

To compress the Documents table, you can use the following SQL query:
要压缩 Documents 表格，您可以使用以下 SQL 查询：

```sql
VACUUM FULL Documents;
```

This command will rebuild the Documents table and reclaim any unused space.
此命令将重建 Documents 表格并回收任何未使用的空间。

This can take a long time, so it is recommended that you do this during off-peak hours.
这可能需要很长时间，因此建议在非高峰时段执行此操作。

### Conclusion
### 结论

By following these steps, you can clean up the Documents table and make it easier to manage.
通过遵循这些步骤，您可以清理 Documents 表格，使其更易于管理。

In [14]:
# Clean up KDB.AI "documents" table and index for similarity search
# so this notebook could be played again and again
session.table("documents").drop()

True